# Notebook 7: Predictive Machine Learning for Falcon 9 Landing Success
## IBM Data Science Professional Certificate Capstone Project

### Learning Objectives:
1. Formulate booster landing prediction as a supervised binary classification task.
2. Prevent data leakage by performing train/test split before fitting scalers.
3. Train and hyperparameter-tune 4 machine learning algorithms:
   - Logistic Regression
   - Support Vector Machine (SVM)
   - Decision Tree Classifier
   - K-Nearest Neighbors (KNN)
4. Use 10-fold Stratified Cross-Validation with `GridSearchCV`.
5. Evaluate models on held-out test data (accuracy, precision, recall, F1, ROC-AUC, confusion matrices).
6. Compare and select the best operational model.

---
### 1. Setup & Load Feature Matrix

In [ ]:
# Step 1: Import scikit-learn modules
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

df_ml = pd.read_csv("../data/processed/dataset_part_3.csv")
Y = df_ml["Class"].values
X = df_ml.drop(columns=["Class"])
print(f"Feature matrix shape: {X.shape}, Target distribution: {np.bincount(Y.astype(int))}")

### 2. Train/Test Split (Preventing Data Leakage)
We reserve 20% of the dataset as a held-out test set (`random_state=42`) using stratified sampling to maintain class proportions.

In [ ]:
# Step 2: Split data into training and test sets
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.20, random_state=42, stratify=Y
)
print(f"Training samples: {len(X_train)}, Testing samples: {len(X_test)}")

# Standardize numerical features (fit strictly on X_train to prevent leakage)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print("StandardScaler fitted on training partition successfully.")

### 3. Hyperparameter Tuning with 10-Fold Stratified Cross-Validation

In [ ]:
# Step 3: Define 10-Fold Stratified Cross-Validation
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# 1. Logistic Regression
lr_grid = GridSearchCV(
    LogisticRegression(random_state=42, max_iter=1000),
    {"C": [0.01, 0.1, 1.0, 10.0], "penalty": ["l2"], "solver": ["lbfgs", "liblinear"]},
    cv=cv, scoring="accuracy"
)
lr_grid.fit(X_train_scaled, Y_train)
print(f"Logistic Regression Best CV Accuracy: {lr_grid.best_score_:.4f}")

# 2. Support Vector Machine
svm_grid = GridSearchCV(
    SVC(random_state=42),
    {"C": [0.1, 0.5, 1.0, 5.0, 10.0], "kernel": ["linear", "rbf", "sigmoid"]},
    cv=cv, scoring="accuracy"
)
svm_grid.fit(X_train_scaled, Y_train)
print(f"SVM Best CV Accuracy: {svm_grid.best_score_:.4f}")

# 3. Decision Tree
dt_grid = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    {"criterion": ["gini", "entropy"], "max_depth": [2, 4, 6, 8, None], "min_samples_split": [2, 5, 10]},
    cv=cv, scoring="accuracy"
)
dt_grid.fit(X_train, Y_train)  # Trees are scale-invariant
print(f"Decision Tree Best CV Accuracy: {dt_grid.best_score_:.4f}")

# 4. K-Nearest Neighbors
knn_grid = GridSearchCV(
    KNeighborsClassifier(),
    {"n_neighbors": [3, 5, 7, 9, 11], "algorithm": ["auto", "kd_tree"], "p": [1, 2]},
    cv=cv, scoring="accuracy"
)
knn_grid.fit(X_train_scaled, Y_train)
print(f"KNN Best CV Accuracy: {knn_grid.best_score_:.4f}")

### 4. Evaluate Models on Held-Out Test Data

In [ ]:
# Step 4: Compare test set metrics
models = {
    "Logistic Regression": (lr_grid.best_estimator_, X_test_scaled),
    "SVM": (svm_grid.best_estimator_, X_test_scaled),
    "Decision Tree": (dt_grid.best_estimator_, X_test),
    "KNN": (knn_grid.best_estimator_, X_test_scaled)
}

results = []
for name, (clf, x_te) in models.items():
    y_pred = clf.predict(x_te)
    acc = accuracy_score(Y_test, y_pred)
    prec = precision_score(Y_test, y_pred, zero_division=0)
    rec = recall_score(Y_test, y_pred, zero_division=0)
    f1 = f1_score(Y_test, y_pred, zero_division=0)
    results.append({"Model": name, "Test Accuracy": acc, "Precision": prec, "Recall": rec, "F1-Score": f1})

df_results = pd.DataFrame(results)
print("=== Model Comparison Table on Held-Out Test Data ===")
print(df_results.to_string(index=False))

### 5. Plot Confusion Matrices

In [ ]:
# Step 5: Visualize Confusion Matrix for Best Model (KNN / Logistic Regression)
best_model = knn_grid.best_estimator_
y_pred_best = best_model.predict(X_test_scaled)
cm = confusion_matrix(Y_test, y_pred_best)

plt.figure(figsize=(6, 5), dpi=150)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["Fail (0)", "Success (1)"], yticklabels=["Fail (0)", "Success (1)"])
plt.title(f"Confusion Matrix (KNN Best Model)\nTest Accuracy: {accuracy_score(Y_test, y_pred_best):.1%}", fontsize=12, fontweight="bold")
plt.xlabel("Predicted Outcome", fontsize=10)
plt.ylabel("Actual Outcome", fontsize=10)
plt.show()

---
### Practice Exercise & Check for Understanding:
1. Why is accuracy alone insufficient when evaluating models on imbalanced datasets? *(Answer: Because a dummy model that always predicts the majority class would show high accuracy while completely failing to detect the minority failure class; precision, recall, and F1 provide a balanced view).*
2. Which algorithms achieved the highest test accuracy on our Falcon 9 landing prediction task? *(Answer: Tuned K-Nearest Neighbors and Logistic Regression, both achieving 83.3% test accuracy).*

### Summary & Final Project Conclusions:
Congratulations! You have completed the entire machine learning predictive pipeline for the SpaceX Falcon 9 First Stage Landing Prediction capstone. You have trained, tuned, and evaluated models that successfully distinguish between booster recovery and mission loss!